# Job-Skills Matching with Cosine Similarity (Colab → Qdrant → FastAPI → Deploy)

This notebook walks you through:
- Creating sentence embeddings for job skills using sentence-transformers (no heavy training needed).
- Uploading vectors to Qdrant Cloud (free tier) for cosine similarity search.
- Building a FastAPI microservice for matching user questionnaire answers to best-fit skills.
- Containerizing and deploying to Azure Container Apps (recommended) or Railway.

Why this approach fits your use-case:
- You’re not training a deep model; you’re embedding texts and doing cosine similarity.
- Colab handles the one-time embedding job for your job-skills corpus.
- Qdrant handles fast vector search for free.
- Your Laravel app just calls the FastAPI service for predictions.


In [ ]:
# 1) Set Up Environment and Dependencies

# If running in Colab, this installs required packages here. On local VS Code, use a venv.
# GPU is not required for this workflow.
!pip -q install sentence-transformers qdrant-client pandas tqdm python-dotenv fastapi uvicorn pydantic numpy scikit-learn faiss-cpu httpx pytest docker

# Verify runtime info
import torch, platform
print("Python:", platform.python_version())
print("Torch CUDA available:", torch.cuda.is_available())

In [ ]:
# 2) Config and Secrets (.env)

# In Colab, set environment variables directly in-session. For local dev, create a .env file.
# Required:
# - QDRANT_URL: https://<your-cluster>.qdrant.io (or dedicated endpoint)
# - QDRANT_API_KEY: your Qdrant API key
# Optional DB (if you also want to persist skills in a relational DB):
# - DB_URL: sqlite:///skills.db (or postgres://...)

import os
from dotenv import load_dotenv

# You can set variables here for a demo. Replace with your real values or use Colab secrets.
# os.environ["QDRANT_URL"] = "https://YOUR-CLUSTER.qdrant.io"
# os.environ["QDRANT_API_KEY"] = "YOUR_KEY"
# os.environ["DB_URL"] = "sqlite:///skills.db"

load_dotenv()  # Will load from .env if present

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
DB_URL = os.getenv("DB_URL", "sqlite:///skills.db")

print("QDRANT_URL:", QDRANT_URL)
print("DB_URL:", DB_URL)

In [ ]:
# 3) Load and Normalize Job Skills Data

import pandas as pd

# Option A: Upload a CSV to Colab (from UI) and set path below
CSV_PATH = None  # e.g., '/content/job_skills.csv'

if CSV_PATH and os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    # Fallback demo data — replace with your real dataset
    df = pd.DataFrame([
        {"skill_id": 1, "title": "Data Analysis", "description": "Analyze datasets and derive insights."},
        {"skill_id": 2, "title": "Python Programming", "description": "Write efficient Python code for data tasks."},
        {"skill_id": 3, "title": "Project Management", "description": "Plan and execute projects effectively."},
        {"skill_id": 4, "title": "Customer Support", "description": "Assist customers and resolve issues."},
    ])

# Normalize text: lowercase, strip, and combine fields for embedding
for col in ["title", "description"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

df["text"] = (df["title"].str.lower() + " | " + df["description"].str.lower()).str.strip()

# Drop duplicates by text
before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
after = len(df)
print(f"Loaded {after} skills (deduped from {before}). Preview:")
df.head()

In [ ]:
# 4) Initialize Database (optional local SQLite/Postgres)

from sqlalchemy import create_engine, Column, Integer, String, Text, LargeBinary, Float
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()

class Skill(Base):
    __tablename__ = 'skills'
    id = Column(Integer, primary_key=True)
    title = Column(String(255), nullable=False)
    description = Column(Text, nullable=True)
    text = Column(Text, nullable=False)

class SkillEmbedding(Base):
    __tablename__ = 'skill_embeddings'
    skill_id = Column(Integer, primary_key=True)
    dim = Column(Integer, nullable=False)
    # store as binary; for Postgres you can use vector type via pgvector instead
    vector = Column(LargeBinary, nullable=False)

engine = create_engine(DB_URL, echo=False)
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine)

print("Database initialized:", DB_URL)